In [1]:
import pandas as pd

# Load Artist related data
artists = pd.read_parquet('./data/mb_artist.parquet')
artist_tags = pd.read_parquet('./data/mb_artist_tag.parquet') # Updated name
artist_ratings = pd.read_parquet('./data/mb_artist_ratings.parquet')

# Load Album related data
albums = pd.read_parquet('./data/mb_album.parquet')
album_tags = pd.read_parquet('./data/mb_album_tag_map.parquet')
album_ratings = pd.read_parquet('./data/mb_album_ratings.parquet')

# Verify the loads
dataframes = {
    "Artists": artists, 
    "Artist Tags": artist_tags, 
    "Artist Ratings": artist_ratings,
    "Albums": albums, 
    "Album Tags": album_tags, 
    "Album Ratings": album_ratings
}

for name, df in dataframes.items():
    print(f"✅ {name}: {df.shape[0]:,} rows loaded.")

✅ Artists: 2,867,969 rows loaded.
✅ Artist Tags: 731,552 rows loaded.
✅ Artist Ratings: 77,571 rows loaded.
✅ Albums: 2,241,402 rows loaded.
✅ Album Tags: 3,042,637 rows loaded.
✅ Album Ratings: 143,733 rows loaded.


In [2]:
# Join Artists with Tags
artist_tag_joined = pd.merge(
    artists, 
    artist_tags, 
    left_on='id', 
    right_on='artist_id', 
    how='inner'
)

# Join with Ratings (Left join to keep artists even if unrated)
final_artist_df = pd.merge(
    artist_tag_joined,
    artist_ratings,
    on='artist_id',
    how='left'
).drop(columns=['artist_id']) # Clean up duplicate ID

# Fill missing ratings so math doesn't break
final_artist_df['rating'] = final_artist_df['rating'].fillna(0)
final_artist_df['rating_count'] = final_artist_df['rating_count'].fillna(0)

In [3]:
# Join Albums with Tags
album_tag_joined = pd.merge(
    albums, 
    album_tags, 
    left_on='id', 
    right_on='album_id', 
    how='inner'
)

# Join with Album Ratings
final_album_df = pd.merge(
    album_tag_joined,
    album_ratings,
    on='album_id',
    how='left'
).drop(columns=['album_id'])

# Fill missing ratings
final_album_df['rating'] = final_album_df['rating'].fillna(0)
final_album_df['rating_count'] = final_album_df['rating_count'].fillna(0)

In [5]:
# Rename artist columns for clarity in the master table
artists_prep = final_artist_df.rename(columns={
    'name': 'artist_name',
    'gid': 'artist_gid',
    'area': 'artist_area',
    'tag_id': 'artist_tag_id',
    'tag_count': 'artist_tag_count',
    'rating': 'artist_rating',
    'rating_count': 'artist_rating_count'
})

# Final Join: Link Albums to their Artists
master_df = pd.merge(
    final_album_df,
    artists_prep,
    left_on='artist_credit', # The ID of the artist on the album record
    right_on='id',           # The ID in the artist table
    how='inner',
    suffixes=('_album', '_artist_meta')
).drop(columns=['id_artist_meta'])

# Final rename for album-specific columns
master_df = master_df.rename(columns={
    'id_album': 'album_id',
    'name': 'album_name',
    'tag_id': 'album_tag_id',
    'tag_count': 'album_tag_count',
    'rating': 'album_rating',
    'rating_count': 'album_rating_count'
})

print(f"\n🚀 Master DataFrame ready with {master_df.shape[0]:,} rows.")


🚀 Master DataFrame ready with 157,945,861 rows.
